# ⚡ Sentinel Terminal — Google Colab Fine-Tuning Pipeline

Fine-tune **Qwen2.5-Coder-3B-Instruct** (or 7B) on your personal terminal interactions, canonical TLDR CLI recipes, deterministic error remediation rules, and **5 open-source Hugging Face shell & bash datasets**.

- **Hardware**: Runs on a **Free Google Colab T4 GPU** (16GB VRAM) or A100.
- **Duration**: ~5 to 10 minutes total for SFT + DPO alignment.
- **Output**: LoRA adapter + GGUF model ready for 1-click download back to Sentinel Terminal (`~/.sentinel/models/`).

## 1. Verify GPU Acceleration

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU:      {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:      {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ Please go to Runtime -> Change runtime type -> Select T4 GPU")

## 2. Install High-Speed ML Libraries (Unsloth & TRL)

In [ ]:
%%capture
!pip install --quiet "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --quiet --no-deps trl peft accelerate bitsandbytes
!pip install --quiet datasets transformers

## 3. Upload & Unpack Local Sentinel Dataset

Upload the `sentinel_training_package.zip` generated on your Mac.

In [ ]:
import os, zipfile
from google.colab import files

if not os.path.exists("sentinel_training_package.zip") and not os.path.exists("sentinel_sft_dataset.jsonl"):
    print("Upload your 'sentinel_training_package.zip' from your Mac:")
    uploaded = files.upload()

if os.path.exists("sentinel_training_package.zip"):
    with zipfile.ZipFile("sentinel_training_package.zip", "r") as z:
        z.extractall(".")

print("\n✓ Local Sentinel files:")
!ls -lh *.jsonl

## 3.5. (Recommended) Enrich with 5 Open-Source Hugging Face Shell & Bash Datasets

Pulls thousands of verified English-to-Bash commands from top open-source repositories:
1. **`emirkaanozdemr/bash_command_data_6K`**: 6,000 natural language to Unix commands
2. **`b-mc2/cli-commands-explained`**: 5,000+ commands from Commandlinefu and TLDR
3. **`aelhalili/bash-commands-dataset`**: Real-world sysadmin and DevOps tasks
4. **`mrheinen/linux-commands`**: Practical Linux system administration recipes
5. **`AmanPriyanshu/tool-reasoning-sft-CODING-text_to_terminal_v2`**: Advanced tool reasoning traces

In [ ]:
# Set to True to enrich training with open-source Hugging Face bash datasets
INCLUDE_HF_DATASETS = True
MAX_SAMPLES_PER_DATASET = 1000  # Pull up to 5,000 extra verified bash commands

SYSTEM_PROMPT = 'You are Sentinel, an autonomous on-device terminal copilot on macOS and Linux. You directly execute verified shell commands. Output strictly valid JSON matching: {"action": "execute", "command": "<cmd>", "explanation": "<brief reason>"}.'

if INCLUDE_HF_DATASETS:
    from datasets import load_dataset
    import json
    
    hf_augmented = []
    print("⚡ Downloading and formatting open-source shell datasets from Hugging Face...")
    
    # 1. emirkaanozdemr/bash_command_data_6K
    try:
        ds1 = load_dataset("emirkaanozdemr/bash_command_data_6K", split="train")
        count = 0
        for row in ds1:
            p = row.get("Instruction") or row.get("instruction")
            c = row.get("Command") or row.get("command")
            if p and c and len(c.strip()) > 1:
                hf_augmented.append({
                    "messages": [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": p.strip()},
                        {"role": "assistant", "content": json.dumps({"action": "execute", "command": c.strip(), "explanation": f"Execute: {c.strip()[:60]}"})}
                    ]
                })
                count += 1
                if count >= MAX_SAMPLES_PER_DATASET: break
        print(f"✓ Loaded {count} samples from emirkaanozdemr/bash_command_data_6K")
    except Exception as e:
        print(f"  Notice on ds1: {e}")
    
    # 2. b-mc2/cli-commands-explained
    try:
        ds2 = load_dataset("b-mc2/cli-commands-explained", split="train")
        count = 0
        for row in ds2:
            p = row.get("summary") or row.get("description")
            c = row.get("command")
            exp = row.get("explanation") or "Execute CLI command"
            if p and c and len(c.strip()) > 1:
                hf_augmented.append({
                    "messages": [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": p.strip()},
                        {"role": "assistant", "content": json.dumps({"action": "execute", "command": c.strip(), "explanation": str(exp).strip()[:80]})}
                    ]
                })
                count += 1
                if count >= MAX_SAMPLES_PER_DATASET: break
        print(f"✓ Loaded {count} samples from b-mc2/cli-commands-explained")
    except Exception as e:
        print(f"  Notice on ds2: {e}")
    
    # 3. aelhalili/bash-commands-dataset
    try:
        ds3 = load_dataset("aelhalili/bash-commands-dataset", split="train")
        count = 0
        for row in ds3:
            p = row.get("prompt") or row.get("instruction")
            c = row.get("command") or row.get("bash")
            if p and c and len(c.strip()) > 1:
                hf_augmented.append({
                    "messages": [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": p.strip()},
                        {"role": "assistant", "content": json.dumps({"action": "execute", "command": c.strip(), "explanation": "Sysadmin automation command"})}
                    ]
                })
                count += 1
                if count >= MAX_SAMPLES_PER_DATASET: break
        print(f"✓ Loaded {count} samples from aelhalili/bash-commands-dataset")
    except Exception as e:
        print(f"  Notice on ds3: {e}")
    
    # 4. mrheinen/linux-commands
    try:
        ds4 = load_dataset("mrheinen/linux-commands", split="train")
        count = 0
        for row in ds4:
            p = row.get("description") or row.get("prompt")
            c = row.get("command")
            if p and c and len(c.strip()) > 1:
                hf_augmented.append({
                    "messages": [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": p.strip()},
                        {"role": "assistant", "content": json.dumps({"action": "execute", "command": c.strip(), "explanation": p.strip()[:80]})}
                    ]
                })
                count += 1
                if count >= MAX_SAMPLES_PER_DATASET: break
        print(f"✓ Loaded {count} samples from mrheinen/linux-commands")
    except Exception as e:
        print(f"  Notice on ds4: {e}")
    
    # 5. AmanPriyanshu/tool-reasoning-sft-CODING-text_to_terminal_v2
    try:
        ds5 = load_dataset("AmanPriyanshu/tool-reasoning-sft-CODING-text_to_terminal_v2-sft-tool-use-agent-data-cleaned-rectified", split="train")
        count = 0
        for row in ds5:
            p = row.get("prompt") or row.get("question")
            c = row.get("command") or row.get("output")
            if p and c and not str(c).startswith("{") and len(str(c)) < 250:
                hf_augmented.append({
                    "messages": [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": str(p).strip()},
                        {"role": "assistant", "content": json.dumps({"action": "execute", "command": str(c).strip(), "explanation": "Tool reasoning execution step"})}
                    ]
                })
                count += 1
                if count >= MAX_SAMPLES_PER_DATASET: break
        print(f"✓ Loaded {count} samples from AmanPriyanshu/tool-reasoning-sft")
    except Exception as e:
        print(f"  Notice on ds5: {e}")
    
    print(f"\n🔥 Total open-source Bash samples added: {len(hf_augmented)}")
    with open("sentinel_sft_dataset.jsonl", "a", encoding="utf-8") as f:
        for item in hf_augmented:
            f.write(json.dumps(item) + "\n")
    print("✓ Appended to sentinel_sft_dataset.jsonl successfully!")

## 4. Load Base Model with 4-bit Quantization (Qwen2.5-Coder-3B)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto detection
load_in_4bit = True # 4-bit quantization for free Colab T4 GPU

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-Coder-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Inject fast LoRA adapter
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0.0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✓ Model and LoRA adapter initialized successfully!")

## 5. Supervised Fine-Tuning (SFT)

Teaches the model Sentinel's shell execution contract, canonical recipes, and broad Unix workflows.

In [ ]:
import json
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

with open("sentinel_sft_dataset.jsonl", "r", encoding="utf-8") as f:
    sft_raw = [json.loads(line) for line in f if line.strip()]

formatted_samples = []
for item in sft_raw:
    text = tokenizer.apply_chat_template(item["messages"], tokenize=False, add_generation_prompt=False)
    formatted_samples.append({"text": text})

sft_dataset = Dataset.from_list(formatted_samples)
print(f"✓ Total training dataset size: {len(sft_dataset)} samples.")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = sft_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 120,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

trainer_stats = trainer.train()
print("\n✓ SFT Training complete!")

## 6. Direct Preference Optimization (DPO)

Aligns the model to violently penalize conversational refusals (*"I cannot access your system..."*) and boost direct, non-destructive shell execution.

In [ ]:
from trl import DPOTrainer, DPOConfig

with open("sentinel_dpo_dataset.jsonl", "r", encoding="utf-8") as f:
    dpo_raw = [json.loads(line) for line in f if line.strip()]

dpo_dataset = Dataset.from_list([
    {"prompt": d["prompt"], "chosen": d["chosen"], "rejected": d["rejected"]}
    for d in dpo_raw
])
print(f"✓ Loaded {len(dpo_dataset)} DPO preference pairs.")

dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,
    tokenizer = tokenizer,
    train_dataset = dpo_dataset,
    args = DPOConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        max_steps = 40,
        learning_rate = 5e-5,
        beta = 0.1,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        output_dir = "dpo_outputs",
        report_to = "none",
    ),
    max_length = max_seq_length,
)

dpo_trainer.train()
print("\n✓ DPO Alignment complete!")

## 6.5. 🧪 Live Benchmark: Test Real-World Terminal Performance

Compare the trained model against known real-world edge cases:
- **Test 1 (Passes in Fine-Tuned, Fails in Normal Model)**: `"kill whatever is running on port 3000"` (Base produces conversational prose or Linux `fuser`; Fine-tuned outputs strictly valid JSON `lsof -ti:3000 | xargs kill -9`).
- **Test 2 (Passes in Fine-Tuned, Fails in Base)**: `"flush dns cache on my mac"` (Base suggests Linux `systemd-resolve`; Fine-tuned outputs macOS `dscacheutil`).
- **Test 3 (Fails in BOTH, but within syllabus)**: `"in-place replace all occurrences of 'localhost' with '127.0.0.1' in app.json using sed"` (Both models emit GNU sed `sed -i 's/...' file`, which triggers fatal syntax error on macOS BSD sed because `-i` requires a backup extension `sed -i ''`).

In [ ]:
# Switch model to ultra-fast inference mode
FastLanguageModel.for_inference(model)

test_prompts = [
    ("Passes in Fine-Tuned, Fails in Base (Port Kill)", "kill whatever is running on port 3000"),
    ("Passes in Fine-Tuned, Fails in Base (macOS DNS Flush)", "flush dns cache on my mac"),
    ("Fails in BOTH (macOS BSD sed -i syntax trap)", "in-place replace all occurrences of 'localhost' with '127.0.0.1' in app.json using sed")
]

print("=" * 75)
print("⚡ LIVE SENTINEL INFERENCE BENCHMARK")
print("=" * 75)

for category, prompt in test_prompts:
    print(f"\n🔹 Test Category: {category}")
    print(f"   User Input:    \"{prompt}\"")
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(input_ids=inputs, max_new_tokens=120, temperature=0.1, use_cache=True)
        response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    
    print(f"   Model Output:  {response.strip()}")
    try:
        parsed = json.loads(response.strip())
        print(f"   Status:        ✅ Valid JSON | Action: {parsed.get('action')} | Cmd: {parsed.get('command')}")
    except Exception:
        print(f"   Status:        ❌ Non-JSON or Conversational Output")
    print("-" * 75)


## 7. Export LoRA Adapter & GGUF Model

In [ ]:
# 1. Save standard LoRA adapter
model.save_pretrained("sentinel_lora_colab")
tokenizer.save_pretrained("sentinel_lora_colab")

# 2. Save 4-bit GGUF model for direct llama-server execution
model.save_pretrained_gguf("sentinel_gguf", tokenizer, quantization_method = "q4_k_m")

print("✓ Exported artifacts:")
!ls -lh sentinel_gguf/

## 8. Download Trained Artifacts to Your Mac

In [ ]:
from google.colab import files
import glob

# Zip and download LoRA adapter
!zip -r sentinel_lora_colab.zip sentinel_lora_colab/
files.download("sentinel_lora_colab.zip")

# Download GGUF binary if created
ggufs = glob.glob("sentinel_gguf/*.gguf")
if ggufs:
    print(f"Downloading {ggufs[0]}...")
    files.download(ggufs[0])

print("\n🎉 Done! To deploy in Sentinel Terminal on your Mac:")
print("  1. Extract sentinel_lora_colab.zip into ~/.sentinel/models/sentinel_mlx_lora")
print("  2. Or copy the .gguf file to ~/.sentinel/models/")
print("  3. Sentinel will automatically hot-reload the fine-tuned adapter!")